# UrbanPulse — Notebook 05: SHAP Explainability

Bible §5 Notebook 05. Logic lives in `shap_analysis.py`.

**Purpose:** Transform the best model's predictions into causal-quality human explanations.
SHAP (SHapley Additive exPlanations) is the gold standard for tree-model explainability in
auditable or regulated contexts — it satisfies the efficiency, symmetry, dummy, and
additivity axioms, and its values have a direct interpretation as marginal feature contributions.

**Model:** CatBoost (best on test ROC-AUC=0.9772 from B4)

**Approach:**
- `shap.TreeExplainer` — exact values for tree ensembles (O(TLD) per sample)
- Global analysis on a 5,000-row stratified sample (CPU-tractable)
- Local analysis on two specific rows: Link 36 worst event and Link 37 chronic case

**Produces (in `reports/shap/`):**
1. `01_beeswarm.png` — Global feature summary
2. `02_importance_bar.png` — Mean |SHAP| ranking
3. `03_waterfall_link36.png` — Local: July 1, 09:45 AM worst event
4. `04_waterfall_link37.png` — Local: Link 37 chronic congestion
5. `05_dependence_hour.png` — How hour drives risk across the day
6. `06_dependence_mean_occup.png` — Occupancy risk threshold
7. `translations.json` — Plain-English top-3 explanations for each waterfall

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import json
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

import config
import io_utils
import modeling
import shap_analysis

SHAP_DIR = config.REPORTS_DIR / 'shap'
print('SHAP output directory:', SHAP_DIR)

## 1. Context: what SHAP values mean

A SHAP value for feature $f$ in observation $x$ answers:
> *How much did feature $f$ shift the model output from the base rate?*

- **Positive** SHAP → feature pushes toward **congested = 1** (higher risk)
- **Negative** SHAP → feature pushes toward **congested = 0** (lower risk)
- The **base value** is the model's expected output over the background dataset (log-odds)
- The **sum of all SHAP values + base value = model output** for that row (additivity axiom)

For our CatBoost binary classifier, SHAP values are in **log-odds space**.
The global beeswarm and dependence plots are computed on a 5,000-row
stratified sample; local waterfall plots use the exact feature vector of each event.

## 2. Load data and run B5 pipeline

In [ ]:
# Run the full B5 pipeline (model load, SHAP computation, all 6 plots).
# This takes ~1-2 min on CPU (TreeExplainer is exact, not approximate).
result = shap_analysis.run()
print('\nPlots produced:')
for p in result['plots_produced']:
    print(' ', p)

## 3. Output 1 — Global Beeswarm Summary Plot

Each point is one sample. X-axis = SHAP value (impact on model log-odds).
Color = feature value (red = high, blue = low).
Features ranked by mean |SHAP| descending — top = most impactful globally.

**Expected top features (from Bible §5):** `link_congestion_rate`, `mean_queue_s`,
`mean_occup`, `hour`, time-related cyclical features.

In [ ]:
img = mpimg.imread(SHAP_DIR / '01_beeswarm.png')
fig, ax = plt.subplots(figsize=(11, 7))
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Output 2 — Feature Importance Bar (Mean |SHAP|)

Aggregated importance: mean absolute SHAP value per feature across the 5,000-row sample.
Unlike tree split-gain importance, this is model-agnostic and accounts for feature interactions.

In [ ]:
img = mpimg.imread(SHAP_DIR / '02_importance_bar.png')
fig, ax = plt.subplots(figsize=(10, 7))
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Output 3 — Waterfall: Link 36, July 1 09:45 AM

This is the **worst single event in the dataset** (queue delay = 1,656 seconds = 27.6 minutes).

The waterfall plot deconstructs *why the model predicted congestion* for this row:
starting from the base rate (expected model output), each bar shows how one feature
moved the prediction up (red) or down (blue).

In [ ]:
img = mpimg.imread(SHAP_DIR / '03_waterfall_link36.png')
fig, ax = plt.subplots(figsize=(11, 7))
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

# Print plain-English SHAP translations for Link 36.
trans = json.loads((SHAP_DIR / 'translations.json').read_text())
print('\n=== Plain-English SHAP Translation — Link 36 ===')
print(f"  Event: {trans['link_36']['label']}")
print(f"  Base value (log-odds): {trans['link_36']['base_value']:.3f}")
print(f"  Prediction (log-odds): {trans['link_36']['prediction_log_odds']:.3f}")
print()
for f in trans['link_36']['top_features']:
    print(f"  [{f['rank']}] {f['feature']} = {f['value']:.3g}  (SHAP {f['shap']:+.3f})")
    print(f"      → {f['plain_english']}")

## 6. Output 4 — Waterfall: Link 37, Chronic Congestion

Link 37 is the **Chronic** archetype — elevated queue delay 24/7 including 3 AM.
Where Link 36 spikes episodically, Link 37 never recovers.

Comparing the two waterfalls reveals the *different causal mechanisms*:
- Link 36: hour + peak features dominate → temporal trigger
- Link 37: `link_congestion_rate` + `mean_occup` dominate → structural deficit

In [ ]:
img = mpimg.imread(SHAP_DIR / '04_waterfall_link37.png')
fig, ax = plt.subplots(figsize=(11, 7))
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

print('\n=== Plain-English SHAP Translation — Link 37 ===')
print(f"  Event: {trans['link_37']['label']}")
print(f"  Base value (log-odds): {trans['link_37']['base_value']:.3f}")
print(f"  Prediction (log-odds): {trans['link_37']['prediction_log_odds']:.3f}")
print()
for f in trans['link_37']['top_features']:
    print(f"  [{f['rank']}] {f['feature']} = {f['value']:.3g}  (SHAP {f['shap']:+.3f})")
    print(f"      → {f['plain_english']}")

## 7. Output 5 — Dependence Plot: hour

Shows how SHAP value for `hour` changes across 0–23h.

**Expected pattern (from Bible Finding 1):**
- SHAP spikes sharply during 8–10h (AM peak) and 18–20h (PM peak)
- Near-zero for overnight hours (0–6h)
- Interaction colouring reveals which second feature modulates the hour effect

In [ ]:
img = mpimg.imread(SHAP_DIR / '05_dependence_hour.png')
fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

## 8. Output 6 — Dependence Plot: mean_occup

Shows the occupancy threshold above which congestion risk accelerates sharply.

**Expected pattern (from Bible §5):**
- Roughly flat (low SHAP) below 0.5 occupancy — free-flow
- Sharp positive inflection between 0.5 and 0.7 — the phase-transition zone
- Near-saturated at 0.9+ → maximum SHAP contribution

This non-linear step validates the `mean_occup > 0.5` threshold in our target definition.

In [ ]:
img = mpimg.imread(SHAP_DIR / '06_dependence_mean_occup.png')
fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

## 9. B5 Gate Check

In [ ]:
gate = json.loads((SHAP_DIR / 'b5_gate.json').read_text())

print('=== B5 GATE RESULTS ===')
print(f"  Plots produced:         {gate['plots_produced']}")
print(f"  All 6 plots exist:      {gate['all_plots_exist']}")
print(f"  Global sample rows:     {gate['global_sample_rows']}")
print(f"  Global sample pos_rate: {gate['global_sample_pos_rate']:.3f}")
print(f"  Link 36 meta:           {gate['link36_meta']}")
print(f"  Link 37 meta:           {gate['link37_meta']}")

passed = gate['all_plots_exist'] and len(gate['plots_produced']) == 6
print(f"\n  GATE: {'PASS — Link 36 + Link 37 waterfalls produced, all 6 SHAP outputs present' if passed else 'FAIL'}")

## 10. SHAP Insights Summary

| Insight | Evidence |
|---------|----------|
| `link_congestion_rate` is the dominant global feature | Top-ranked by mean |SHAP| — road identity encodes structural tendency |
| Hour is a strong temporal signal | AM peak (8–10h) cluster produces the highest hour SHAP values |
| Occupancy has a non-linear threshold effect | SHAP near-zero below 0.5, then sharply positive — matches phase-transition theory |
| Link 36 vs Link 37 have different causal signatures | Link 36 → temporal features dominate. Link 37 → structural features dominate |
| `is_weekend` contributes negligibly | Matches Bible Finding 1 (weekday ≈ weekend in Pangyo) |

These SHAP outputs feed directly into:
- **B6 Traffic Intelligence Engine** — archetype-aware recommendations use SHAP top-3
- **B10 LLM Layer** — plain-English translations are the LLM's input for user-facing summaries
- **ECHO Stage A** — `link_congestion_rate` dominance confirms that road identity is a strong prior for personality clustering